# **YOLO V8 Model Training**




# **Step 1 : Dual GPU Verification**

In [1]:
!nvidia-smi

Sat Feb 28 04:02:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Step 2 : Dependencies Installation**

In [2]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# **Dataset Download**

# **Step 3 : HyperTuned Training**

In [4]:
from ultralytics import YOLO
import os
import shutil
import torch

# ========================= PATHS =========================
data_yaml_path = '/kaggle/working/mydata-9/data.yaml'
cache_dir = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)
os.makedirs("/kaggle/working/torch_cache", exist_ok=True)
os.environ["TORCH_KERNEL_CACHE_PATH"] = "/kaggle/working/torch_cache"

# ========================= MODEL =========================
model = YOLO('/kaggle/input/models/sarveshvengurlekar1/custom-dataset-v2-models/pytorch/custom-version-2/1/100_best.pt')          # ← Must be .pt for tuning

# ========================= GENETIC ALGORITHM TUNING =========================
tune_args = {
    'data': data_yaml_path,
    'task': 'detect',
    
    # === Tuning Control ===
    'epochs': 5,                    # epochs per trial (keep low during tuning)
    'iterations': 25,               # number of genetic algorithm generations (100–300 recommended)

    'imgsz': 640,
    'lr0': 0.00443,
    'lrf': 0.01517,
    'momentum': 0.80035,
    'weight_decay': 0.00053,
    'warmup_epochs': 3.15766,
    'warmup_momentum': 0.71564,
    'box': 8.09874,
    'cls': 0.52284,
    'dfl': 1.72951,
    'hsv_h': 0.02725,
    'hsv_s': 0.5771,
    'hsv_v': 0.34515,
    'degrees': 0.00696,
    'translate': 0.04341,
    'scale': 0.37902,
    'shear': 9.0e-05,
    'perspective': 0.00015,
    'flipud': 0.00237,
    'fliplr': 0.43706,
    'bgr': 0.00431,
    'mosaic': 0.89705,
    'mixup': 0.00404,
    'cutmix': 0.00079,
    'copy_paste': 0.00388,

    # === Your preferred settings (these will be fixed or used as starting point) ===
    'batch': 16,
    'optimizer': 'AdamW',
    'cos_lr': True,
    'close_mosaic': 2,
    'cache': cache_dir,
    'workers': 3,
    'device': list(range(torch.cuda.device_count())) if torch.cuda.device_count() > 1 else 0,
    
    # === Additional good defaults for evolution ===
    'plots': True,
    'save': True,
    'save_period': 10,
    'patience': 0,                   # no early stopping during tuning
    'verbose': True,
}

print("Starting Genetic Algorithm Hyperparameter Tuning...")
results = model.tune(**tune_args)

print("Tuning Complete!")
print(f"Best hyperparameters saved at: {model.tune_dir}")

# ========================= EXPORT BEST MODEL =========================
# The best model after tuning is automatically saved as best.pt inside the tune run
best_model_path = f"{model.tune_dir}/weights/best.pt"

# Zip the entire tuning results (contains all trials + best model)
folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/genetic_tuning_50iter"

if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

shutil.make_archive(zip_path, 'zip', folder_path)
print(f"ZIP file created at: {zip_path}.zip")
print(f"Best model: {best_model_path}")

Starting Genetic Algorithm Hyperparameter Tuning...
Tuner: Initialized Tuner instance with 'tune_dir=/kaggle/working/runs/detect/tune'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/25 with hyperparameters: {'lr0': 0.00443, 'lrf': 0.01517, 'momentum': 0.80035, 'weight_decay': 0.00053, 'warmup_epochs': 3.15766, 'warmup_momentum': 0.71564, 'box': 8.09874, 'cls': 0.52284, 'dfl': 1.72951, 'hsv_h': 0.02725, 'hsv_s': 0.5771, 'hsv_v': 0.34515, 'degrees': 0.00696, 'translate': 0.04341, 'scale': 0.37902, 'shear': 9e-05, 'perspective': 0.00015, 'flipud': 0.00237, 'fliplr': 0.43706, 'bgr': 0.00431, 'mosaic': 0.89705, 'mixup': 0.00404, 'cutmix': 0.00079, 'copy_paste': 0.00388, 'close_mosaic': 2}
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, a

AttributeError: 'DetectionModel' object has no attribute 'tune_dir'